In [24]:
import matplotlib
import jax
from flax import nnx
import jax.numpy as jnp
import math
import optax
import gc

In [25]:
class SelfAttention(nnx.Module):
  def __init__(self, in_features, qKV_features, out_features, rngs: nnx.Rngs):
    self.w_q = nnx.Linear(qKV_features, use_bias=False, rngs=rngs)
    self.w_k = nnx.Linear(qKV_features, use_bias=False, rngs=rngs)
    self.w_v = nnx.Linear(qKV_features, use_bias=False, rngs=rngs)

  def __call__(self, x):
    queries = self.w_q(x)
    keys = self.w_k(x)
    values = self.w_v(x)

    # q.(k.T)
    alignment_scores = queries @ keys.T

    # scaled
    alignment_scores = alignment_scores / math.sqrt(queries.shape[-1])

    # masked, lower triangle set to -inf before softmax
    mask = jnp.tril(jnp.ones(alignment_scores.shape))
    alignment_scores = jnp.where(mask == 0, alignment_scores, -jnp.inf)

    # softmax
    alignment_scores = jax.nn.softmax(alignment_scores)

    # aligment_scores . values
    attention_values = alignment_scores @ values

    return attention_values

In [26]:
class MultiHeadAttention(nnx.Module):
  def __init__(self, n_heads, in_features, rngs: nnx.Rngs):
    self.n_heads = n_heads
    self.head_dim = in_features // n_heads

    # input: Batch, context_len, embed_dim
    # MultiHead Attention
    #   so each head takes slice of the embeddings
    #   for each token in context as input
    #   therefore, for each head dimensions will be
    #   input: Batch, context_len, embed_dim// n_heads


    self.q_proj = nnx.Linear(in_features, in_features, use_bias=False, rngs=rngs)
    self.k_proj = nnx.Linear(in_features, in_features, use_bias=False, rngs=rngs)
    self.v_proj = nnx.Linear(in_features, in_features, use_bias=False, rngs=rngs)

    self.out_proj = nnx.Linear(in_features, in_features, use_bias=False, rngs=rngs)

  def __call__(self, input_ids, attention_mask):
    # print("MultiHeadAttention")
    # print(f"input_ids: {input_ids.shape}")
    # print(f"attention_mask: {attention_mask.shape}")
    x = input_ids * attention_mask

    # print(f"x: {x.shape}")

    # x: Batch, Context len, Embed dim
    B, L, E = x.shape
    assert E == self.n_heads * self.head_dim, f"embed_dim: ({E}), must be divisible by n_heads: ({self.n_heads})"

    queries = self.q_proj(x)
    keys = self.k_proj(x)
    values = self.v_proj(x)

    # print(f"queries: {queries.shape}")
    # print(f"keys: {keys.shape}")
    # print(f"values: {values.shape}")
    queries = queries.reshape(B, L, self.n_heads, self.head_dim)
    keys = keys.reshape(B, L, self.n_heads, self.head_dim)
    values = values.reshape(B, L, self.n_heads, self.head_dim)

    # could use swapaxes, the following reads better to me
    queries = queries.transpose(0, 2, 1, 3)
    keys = keys.transpose(0, 2, 1, 3)
    values = values.transpose(0, 2, 1, 3)

    # print(f"queries: {queries.shape}")
    # print(f"keys: {keys.shape}")
    # print(f"values: {values.shape}")

    alignment_scores = queries @ keys.transpose(0, 1, 3, 2)
    alignment_scores = alignment_scores / math.sqrt(queries.shape[-1])

    mask = jnp.tril(jnp.ones(alignment_scores.shape))
    alignment_scores = jnp.where(mask == 0, alignment_scores, -1e5)

    alignment_scores = jax.nn.softmax(alignment_scores)

    attention_values = alignment_scores @ values
    attention_values = attention_values.transpose(0, 2, 1, 3)

    # print(f"attention_values: {attention_values.shape}")
    attention_values = attention_values.reshape(B, L, E)
    # print(f"attention_values: {attention_values.shape}")

    attention_values = self.out_proj(attention_values)

    # print(f"attention_values: {attention_values.shape}")
    return attention_values

In [27]:
class MLP(nnx.Module):
  def __init__(self, in_features, hidden_features, out_features, rngs: nnx.Rngs):
    self.w1 = nnx.Linear(in_features, hidden_features, rngs=rngs)
    self.w2 = nnx.Linear(hidden_features, out_features, rngs=rngs)

  def __call__(self, x):
    x = jax.nn.gelu(self.w1(x))
    x = self.w2(x)
    return x

In [28]:
class Transformer(nnx.Module):
  def __init__(self, embed_dim, n_heads, mlp_hidden_dim, rngs: nnx.Rngs):
    self.attention = MultiHeadAttention(n_heads, embed_dim, rngs)
    self.mlp = MLP(embed_dim, mlp_hidden_dim, embed_dim, rngs)

    self.ln1 = nnx.LayerNorm(embed_dim, rngs=rngs)
    self.ln2 = nnx.LayerNorm(embed_dim, rngs=rngs)

  def __call__(self, input_ids, attention_mask):
    x = input_ids
    norm_x = self.ln1(x)
    attention = self.attention(norm_x, attention_mask)

    x = x + attention
    norm_x = self.ln2(x)

    mlp_output = self.mlp(norm_x)

    return mlp_output + x


In [29]:
class Model(nnx.Module):
  def __init__(self, embed_dim, seq_len, vocab_size, n_heads, mlp_hidden_dim, n_layers, rngs: nnx.Rngs):
    self.embed_dim = embed_dim
    self.n_heads = n_heads
    self.mlp_hidden_dim = mlp_hidden_dim
    self.n_layers = n_layers

    self.seq_len = seq_len
    self.vocab_size = vocab_size

    self.token_embedding = nnx.Embed(vocab_size, embed_dim, rngs=rngs)
    self.position_embedding = nnx.Embed(seq_len, embed_dim, rngs=rngs)

    self.layers = [Transformer(embed_dim, n_heads, mlp_hidden_dim, rngs=rngs) for _ in range(n_layers)]

    self.ln = nnx.LayerNorm(embed_dim, rngs=rngs)
    self.head = nnx.Linear(embed_dim, vocab_size, use_bias=False, rngs=rngs)

  def __call__(self, input_ids, attention_mask):
    # input_ids: (Batch, SeqLen)
    # attention_mask: (Batch, SeqLen)
    # print(f"model input_ids: {input_ids.shape}")
    # print(f"model attention_mask: {attention_mask.shape}")

    # jax.debug.print("inputs:{iids}",iids=input_ids.astype(jnp.int32).max())
    # jax.debug.print("attn:{att}",att=attention_mask.max())
    attention_mask = jnp.expand_dims(attention_mask, axis=-1)

    t_embeds = self.token_embedding(input_ids.astype(jnp.int32)) # (Batch, SeqLen, EmbedDim)

    # Nans returned; cause, using input_ids.shape[1] here
    p_embeds = self.position_embedding(jnp.arange(input_ids.shape[0])) # (SeqLen, EmbedDim)

    state = nnx.state(self)
    raw_matrix = state.token_embedding.embedding

    # jax.debug.print("{a}", a=jnp.isnan(raw_matrix).any())

    x = t_embeds + p_embeds

    # jax.debug.print("t{t}",t=jnp.isnan(t_embeds).any())
    # jax.debug.print("p{p}",p=jnp.isnan(p_embeds).any())
    # jax.debug.print("x{xm}",xm=jnp.isnan(x).any())

    for layer in self.layers:
      x = layer(x, attention_mask) # Pass embedded sequence 'x' and mask to transformer layer

    # jax.debug.print("afterattentionx{xm}",xm=x.max())

    norm_x = self.ln(x)

    # jax.debug.print("normx{xm}",xm=x.max())

    logits = self.head(norm_x)

    # jax.debug.print("x{xm}",xm=x.max())

    return logits

In [30]:
n_heads = 10
embed_dim = n_heads * 64
seq_len = 50
vocab_size = 50257
mlp_hidden_dim = 1024
n_layers = 10
model = Model(embed_dim=embed_dim, seq_len=seq_len, vocab_size=vocab_size, n_heads=n_heads, mlp_hidden_dim=mlp_hidden_dim, n_layers=n_layers, rngs=nnx.Rngs(42))
# state = nnx.state(model)
# raw_matrix = state.token_embedding.embedding

# print("", jnp.isnan(raw_matrix).any())

In [31]:
nnx.display(model)

In [32]:
!pip install transformers

In [33]:
from transformers import AutoTokenizer as AT

In [34]:
# fetch tiny shakespeare dataset and store in variable for processing
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()
  f.close()

print(f"Dataset len: {len(text)}")
print(text[:100])

--2026-08-05 04:48:06--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2026-08-05 04:48:06 (49.1 MB/s) - ‘input.txt’ saved [1115394/1115394]

Dataset len: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [35]:
from huggingface_hub import login
login()

In [36]:
tokenizer = AT.from_pretrained('gpt2', vocab_size=vocab_size)

In [37]:
inputs = tokenizer(text)
len(inputs)

len(tokenizer)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


50257

In [38]:
len(inputs.input_ids)

338025

In [39]:
len(set(inputs.input_ids))

11706

In [40]:
def append(arr1, arr2, axis=0):
  if len(arr1.shape) == 1:
    return np.array(arr2)

  return np.append(arr1, arr2, axis=axis)

In [41]:
import numpy as np
def generate_batches(inputs, batch_size):
  # inputs is dict with input_ids and attention_mask
  # returns [...input ids..., ... attn masks ...] for each batch
  num_batches = len(inputs.input_ids) // batch_size

  # num_batches = 4
  batches = np.array([])
  for i in range(num_batches):
    batch_input_ids = np.array(inputs.input_ids[i * batch_size : (i + 1) * batch_size])
    batch_attention_mask = np.array(inputs.attention_mask[i * batch_size : (i + 1) * batch_size])

    # print(batch_input_ids.shape)
    # print(batch_attention_mask.shape)

    combined = np.concat((batch_input_ids, batch_attention_mask))

    # print(combined.shape)

    batches = append(batches, [combined])
    # print(batches.shape)
  return batches

In [42]:
batch_size = 100

batches = generate_batches(inputs, batch_size)

print(f"Batches: {batches.shape}")

Batches: (3380, 200)


In [43]:
# train test split the batches
train_batches = batches[:int(len(batches) * 0.9)]
val_batches = batches[int(len(batches) * 0.9):]

print(f"train: {train_batches.shape}")
print(f"val: {val_batches.shape}")

train: (3042, 200)
val: (338, 200)


In [44]:
###########################  Data Manipulation  ################################
def generate_targets_from_input(inputs):
  # takes in batches of input_ids and attention_maskes
  # returns inputs, targets for autoregressive task

  # input_ids, input_attn_masks, target_ids, target_attn_masks

  i_ids = np.array([])
  i_attn_masks = np.array([])

  t_ids = np.array([])
  t_attn_masks = np.array([])
  batch_size = inputs.shape[1] // 2

  for batch in inputs:
    input_ids = batch[:batch_size]
    attention_masks = batch[batch_size:]

    target_ids = jnp.array(input_ids[1:], dtype = jnp.float32)
    input_ids = jnp.array(input_ids[:-1], dtype = jnp.float32)

    target_attention_masks = jnp.array(attention_masks[1:], dtype = jnp.float32)
    input_attention_masks = jnp.array(attention_masks[:-1], dtype = jnp.float32)


    i_ids = append(i_ids, [input_ids], axis=0)
    i_attn_masks = append(i_attn_masks, [input_attention_masks], axis=0)

    t_ids = append(t_ids, [target_ids], axis=0)
    t_attn_masks = append(t_attn_masks, [target_attention_masks], axis=0)

  return i_ids, i_attn_masks, t_ids, t_attn_masks

i_ids, i_attn_masks, t_ids, t_attn_masks = generate_targets_from_input(train_batches)
v_i_ids, v_i_attn_masks, v_t_ids, v_t_attn_masks = generate_targets_from_input(val_batches)

print(f"i_ids: {i_ids.shape}")
print("", i_ids.max())
print(f"i_attn_masks: {i_attn_masks.shape}")

print(f"t_ids: {t_ids.shape}")
print("", t_ids.max())
print(f"t_attn_masks: {t_attn_masks.shape}")

print(f"v_i_ids: {v_i_ids.shape}")
print(f"v_i_attn_masks: {v_i_attn_masks.shape}")

print(f"v_t_ids: {v_t_ids.shape}")

print(f"v_t_attn_masks: {v_t_attn_masks.shape}")

i_ids: (3042, 99)
 50255.0
i_attn_masks: (3042, 99)
t_ids: (3042, 99)
 50255.0
t_attn_masks: (3042, 99)
v_i_ids: (338, 99)
v_i_attn_masks: (338, 99)
v_t_ids: (338, 99)
v_t_attn_masks: (338, 99)


In [45]:
t_inputs = append(i_ids, i_attn_masks)
t_targets = append(t_ids, t_attn_masks)

v_inputs = append(v_i_ids, v_i_attn_masks)
v_targets = append(v_t_ids, v_t_attn_masks)

print(f"inputs: {t_inputs.shape}")
print(f"targets: {t_targets.shape}")
print(f"input:{t_inputs.max()}")
print(f"targets:{t_targets.max()}")
print(f"v_inputs: {v_inputs.shape}")
print(f"v_targets: {v_targets.shape}")

inputs: (6084, 99)
targets: (6084, 99)
input:50255.0
targets:50255.0
v_inputs: (676, 99)
v_targets: (676, 99)


In [46]:
#################### Training ##################################################

@nnx.jit
def train_step(optimizer: nnx.Optimizer, model: Model, inputs, targets):
  ## Loss Fn
  ####
  def loss(model: Model, inputs, targets):
    batch_size = inputs.shape[0] // 2
    i_ids = inputs[:batch_size,:]
    i_attn_masks = inputs[batch_size:,:]
    # jax.debug.print(f"{i_ids.shape}")
    # jax.debug.print(f"{i_attn_masks.shape}")
    logits = model(i_ids, i_attn_masks)

    targets = targets[:batch_size,:]

    # jax.debug.print("logits{l}",l=logits.max())
    # jax.debug.print("targets{t}",t=targets.max())

    targets = targets.astype(jnp.int32)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets)

    # jax.debug.print("{l}", l=jnp.isnan(loss).any())
    # jax.debug.print(f"{jnp.mean(loss, axis=1)[0]}")
    return loss.mean()

  loss_value, grads = jax.value_and_grad(loss)(model, inputs, targets)
  optimizer.update(model, grads)

  return loss_value


#### training loop #####
n_steps = 5

### is it good to have optimizer here if we run this block multiple times in a jupyter
### jupyter notebook?
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=3e-4), wrt=nnx.Param)
training_loss = jnp.array([])
for i in range(n_steps):
  # start_time = jax.process_index() * jax.process_count()
  print(f"Step: {i}")
  batches = t_inputs.shape[0]
  loss_values = jnp.array([])
  for b in range(batches):
    inputs = jnp.array([t_inputs[b], t_inputs[b+batch_size]])
    targets = jnp.array([t_targets[b], t_targets[b+batch_size]])

    loss_value = train_step(optimizer, model, inputs, targets)

    if b % 100 == 0:
      print(f"Batch: {b}, Loss: {loss_value}")
    loss_values = jnp.append(loss_values, loss_value)
  mean_loss = loss_values.mean()
  jnp.append(training_loss, mean_loss)
  print(f"Step: {i}, Loss: {mean_loss}")
  #, Time taken: {jax.process_index()*jax.process_count() - start_time}")

Step: 0
Batch: 0, Loss: 10.992141723632812
Batch: 100, Loss: 6.632117748260498
Batch: 200, Loss: 6.375351428985596
Batch: 300, Loss: 6.1455769538879395
Batch: 400, Loss: 6.310391426086426
Batch: 500, Loss: 7.34182596206665
Batch: 600, Loss: 6.440362453460693
Batch: 700, Loss: 6.4225382804870605
Batch: 800, Loss: 6.793941497802734
Batch: 900, Loss: 6.913475036621094
Batch: 1000, Loss: 6.3738274574279785
Batch: 1100, Loss: 6.702510833740234
Batch: 1200, Loss: 6.594322681427002
Batch: 1300, Loss: 6.462446689605713
Batch: 1400, Loss: 7.151593208312988
Batch: 1500, Loss: 6.369318008422852
Batch: 1600, Loss: 5.9727630615234375
Batch: 1700, Loss: 6.230588436126709
Batch: 1800, Loss: 6.422907829284668
Batch: 1900, Loss: 6.447642803192139
Batch: 2000, Loss: 6.688274383544922
Batch: 2100, Loss: 6.53323221206665
Batch: 2200, Loss: 6.4746222496032715
Batch: 2300, Loss: 7.246695041656494
Batch: 2400, Loss: 6.658341884613037
Batch: 2500, Loss: 6.871266841888428
Batch: 2600, Loss: 6.264082431793213
B

KeyboardInterrupt: 

In [48]:
#############################  Inference  ######################################


def choose_from_logits(logits):
  # print(f"logits during selection: {logits.shape}")
  # logits: (Batch, Seq_len, Vocab_size)
  # from the last tokens predictions
  logits = logits[:, -1, :]
  # select one with max probab
  return jnp.argmax(logits, axis=-1, keepdims=True)

def generate(model, input_ids, n_tokens):
  outputs = []
  attention_mask = jnp.ones_like(input_ids)

  input_ids = jnp.expand_dims(input_ids, axis=0)
  attention_mask = jnp.expand_dims(attention_mask, axis=0)

  for _ in range(n_tokens):
    logits = model(input_ids, attention_mask)
    next_token = choose_from_logits(logits)

    # add to inputs
    inputs = jnp.concatenate([input_ids, next_token], axis=1)

    outputs.append(next_token)

  outputs = jnp.array(outputs).reshape(-1)
  return tokenizer.decode(outputs)

generate(model, v_inputs[0], 10)



'""""""""""'